# HW6 총액 회귀 모델과 API

이종석 · 21011615

같은 폴더의 `sales_clean.csv`를 입력으로 위에서 아래로 모든 셀을 실행합니다.

## ① 입력 읽기와 학습·테스트 분할

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

sales = pd.read_csv("sales_clean.csv")
X = sales[["quantity", "unit_price"]]
y = sales["total_sales"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)
print("training :", len(X_train), " test :", len(X_test))

training : 803  test : 201


## ② 선형 회귀·선형 회귀 + qp·결정 트리 비교

In [2]:
X2_train = X_train.assign(qp=X_train["quantity"] * X_train["unit_price"])
X2_test  = X_test.assign(qp=X_test["quantity"] * X_test["unit_price"])
lin  = LinearRegression().fit(X_train, y_train)
lin2 = LinearRegression().fit(X2_train, y_train)
tree = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)
print("linear regression      test R² :", round(lin.score(X_test, y_test), 3))
print("linear regression + qp test R² :", round(lin2.score(X2_test, y_test), 3))
print("decision tree (no limit) test R² :", round(tree.score(X_test, y_test), 3))

linear regression      test R² : 0.862
linear regression + qp test R² : 1.0
decision tree (no limit) test R² : 0.994


## ③ 선형 회귀에 qp가 필요한 이유

선형 회귀는 `a × quantity + b × unit_price + c`로 예측하므로 두 입력의 곱인 총액을 그대로 표현하지 못해 테스트 R²가 0.862입니다.
`qp = quantity × unit_price`를 추가하면 그 열의 계수 약 1, 나머지 계수와 절편 약 0으로 총액 관계를 표현해 R²가 1.000이 됩니다.

## 단계 ④ ~ ⑤ : 깊이를 고른다

In [3]:
rows = []
for dep in [1, 2, 3, 5, 8, 12, None]:
    m = DecisionTreeRegressor(max_depth=dep, random_state=0).fit(X_train, y_train)
    rows.append([str(dep), m.get_n_leaves(),
                 round(m.score(X_train, y_train), 3), round(m.score(X_test, y_test), 3)])
pd.DataFrame(rows, columns=["max_depth", "leaves", "training R²", "test R²"])

,max_depth,leaves,training R²,test R²
0,1,2,0.361,0.346
1,2,4,0.739,0.711
2,3,8,0.876,0.846
3,5,32,0.972,0.961
4,8,250,0.998,0.991
5,12,781,1.000,0.994
6,None,800,1.000,0.994


In [4]:
for dep in [3, 5, 8, 12, None]:
    s = cross_val_score(DecisionTreeRegressor(max_depth=dep, random_state=0), X, y, cv=5)
    print(f"max_depth {str(dep):4s}  mean {s.mean():.6f}  std {s.std():.6f}")

max_depth 3     mean 0.845258  std 0.019086
max_depth 5     mean 0.960743  std 0.002673
max_depth 8     mean 0.991885  std 0.000397
max_depth 12    mean 0.994620  std 0.000522
max_depth None  mean 0.994721  std 0.000618


## ⑤ 교차검증에 따른 깊이 선택

깊이 8의 교차검증 평균·표준편차는 0.991885·0.000397이고, 깊이 12는 0.994620·0.000522입니다.
평균 차이가 약 0.0027이므로 잎 250개인 깊이 8을 선택합니다. 깊이 12의 잎 수는 781개입니다.
제한 없는 트리는 training R² 1.000, test R² 0.994이며 총액에 실제 곱셈 규칙이 있어 차이가 작게 나타납니다.